# Признаки по марту и поиск правил

In [ ]:
'''
Отбор признаков по переносу на март. CV по фолдам смешивает январь, февраль и март и дважды не угадал Public, а перенос
«обучение на январе и феврале → проверка на марте» оба раза совпал (раунд 2: март +0.017 → Public +0.015; раунд 3: +0.0009 → +0.0009).
Март ближе всего к тесту по времени и сезону. Шум оцениваем парным бутстрепом по клиентам марта, прогноз усреднён по 3 зёрнам.
Шаги:
1. чистка: убираем по одной группы, добавленные в v2–v5, — нет ли среди них мешающих весной;
2. перепроверка отвергнутых по CV групп (сезонность категорий, циклы за год, последний день, t по категориям, повторы);
3. новые группы: весна год назад и история прогнозов (что было с клиентом в прошлых 3-месячных окнах и предсказывала ли их формула t);
4. поиск правил: неглубокие деревья по понятным величинам отдельно по числу заказов, листья с долей покупателей ≥ 0.9 или ≤ 0.05,
   проверенные на марте;
5. итоговый набор: перенос на март, CV и перенос на все срезы, пробный сабмит.
CPU, около 50 минут. Выход: features_march.json (список признаков для v6), submission_march_lgb.csv.
'''
import itertools
import json
import os
import re
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 5
EARLY_STOP = 200
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print("GPU:", GPU)
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)
data["top_category2"] = data["top_category2"].astype("category")
data["top_brand"] = data["top_brand"].astype(object).where(data["top_brand"].notna(), "none").astype(str)

## Признаки по отдельным заказам, интервалам и календарю

In [ ]:
cid = "customer_id"
info = cust.set_index(cid)
LO = pd.Series(float(GAP_DAYS), index=info.index)
HI = (GAP_DAYS + info["win_len"]).astype(float)
EST = ["mean", "median", "min", "max", "q25", "q75", "last", "last2", "last3", "ewm", "trim"]
HIST = {"вся": 10_000, "год": 365, "полгода": 182}


def month_idx(s):
    return s.dt.year * 12 + s.dt.month


def estimators(E):
    E = E.sort_values([cid, "t"]).reset_index(drop=True)
    g = E.groupby(cid)["t"]
    E["gap"] = g.diff()
    E["rk"] = E.groupby(cid).cumcount()
    gp = E[E["gap"].notna()]
    gg = gp.groupby(cid)["gap"]
    out = pd.DataFrame({"r": g.min(), "n": g.size()})
    out["mean"] = (g.max() - g.min()) / (out["n"] - 1).replace(0, np.nan)
    out["median"] = gg.median()
    out["min"] = gg.min()
    out["max"] = gg.max()
    out["q25"] = gg.quantile(0.25)
    out["q75"] = gg.quantile(0.75)
    out["last"] = gp[gp["rk"] == 1].set_index(cid)["gap"]
    out["last2"] = gp[gp["rk"] <= 2].groupby(cid)["gap"].mean()
    out["last3"] = gp[gp["rk"] <= 3].groupby(cid)["gap"].mean()
    w = 0.7 ** (gp["rk"] - 1)
    out["ewm"] = (gp["gap"] * w).groupby(gp[cid]).sum() / w.groupby(gp[cid]).sum()
    out["trim"] = (gg.sum() - gg.max()) / (gg.count() - 1).replace(0, np.nan)
    return out


def unit_feats(E, prefix):
    # «ожидаемый момент следующего заказа» по разным оценкам периода; E — (customer_id, d), d — дней до отсечения
    out = {}
    E = E.assign(t=E["d"].astype(float))
    for hname, H in HIST.items():
        est = estimators(E[E["d"] <= H][[cid, "t"]]).reindex(info.index)
        r = est["r"]
        key = f"{prefix}|{hname}"
        out[f"{key}|n"] = est["n"].fillna(0)
        out[f"{key}|r"] = r
        for e in EST:
            P = est[e].where(est[e] > 0)
            tn, tm = P - r, P - np.mod(r, P)
            pre = f"{key}|{e}"
            out[f"{pre}|t"] = tn
            out[f"{pre}|dist"] = (LO - tn).clip(lower=0) + (tn - HI).clip(lower=0)
            out[f"{pre}|in"] = ((tn > LO) & (tn <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|tm_in"] = ((tm > LO) & (tm <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|k_in"] = np.floor((r + HI) / P) - np.floor((r + LO) / P)
    return out


o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), date=("order_date", "first"), n_items=("paid_price", "size"))
    .reset_index()
    .sort_values([cid, "d", "order_id"])
    .reset_index(drop=True)
)
new = unit_feats(o[[cid, "d"]], "заказ")

# структура интервалов между заказами
o["gap"] = o.groupby(cid)["d"].diff()
gpo = o[o["gap"].notna()]
for name, cond in [("0", gpo["gap"] == 0), ("le1", gpo["gap"] <= 1), ("le3", gpo["gap"] <= 3), ("le7", gpo["gap"] <= 7), ("le14", gpo["gap"] <= 14),
                   ("le30", gpo["gap"] <= 30), ("gt60", gpo["gap"] > 60), ("gt90", gpo["gap"] > 90), ("gt180", gpo["gap"] > 180)]:
    cf = cond.astype(float)
    new[f"интервалы|доля {name}"] = cf.groupby(gpo[cid]).mean()
    new[f"интервалы|доля {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).mean()
    new[f"интервалы|число {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).sum()
dayc = o.groupby([cid, "d"]).agg(k=("order_id", "size"), items=("n_items", "sum")).reset_index()
for w in (90, 365):
    new[f"интервалы|дней с 2+ заказами за {w}"] = ((dayc["k"] >= 2) & (dayc["d"] <= w)).groupby(dayc[cid]).sum()
last_day = dayc.sort_values([cid, "d"]).drop_duplicates(cid).set_index(cid)
new["интервалы|заказов в последний день"] = last_day["k"]
new["интервалы|позиций в последний день"] = last_day["items"]
new["интервалы|давность последнего дня с 2+ заказами"] = dayc[dayc["k"] >= 2].groupby(cid)["d"].min()
r_o = o.groupby(cid)["d"].min()
qs = gpo.groupby(cid)["gap"].quantile([0.25, 0.5, 0.75, 0.9]).unstack()
for q in qs.columns:
    new[f"интервалы|давность / квантиль {q}"] = r_o / qs[q].replace(0, np.nan)
    new[f"интервалы|давность - квантиль {q}"] = r_o - qs[q]
new["интервалы|разброс"] = gpo.groupby(cid)["gap"].std() / gpo.groupby(cid)["gap"].mean().replace(0, np.nan)
pos = gpo[gpo["gap"] > 0].groupby(cid)["gap"]
new["интервалы|средний ненулевой"] = pos.mean()
new["интервалы|давность - средний ненулевой"] = r_o - pos.mean()

# заказы и позиции по непересекающимся интервалам до отсечения
BINS = [0, 7, 14, 30, 60, 90, 120, 180, 270, 365, 450, 540, 733]
labels = [f"{BINS[i]}-{BINS[i + 1]}" for i in range(len(BINS) - 1)]
t_o = pd.crosstab(o[cid], pd.cut(o["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
t_l = pd.crosstab(lines[cid], pd.cut(lines["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
for lab in labels:
    new[f"по интервалам|заказов {lab}"] = t_o[lab]
    new[f"по интервалам|позиций {lab}"] = t_l[lab]
    new[f"по интервалам|доля заказов {lab}"] = t_o[lab] / t_o.sum(axis=1)

# календарь
mi = month_idx(o["date"])
rel = o[cid].map(month_idx(info["scoring_date"])) - mi
last_o = o.drop_duplicates(cid).set_index(cid)
new["календарь|день месяца последнего заказа"] = last_o["date"].dt.day
new["календарь|день недели последнего заказа"] = last_o["date"].dt.dayofweek
new["календарь|месяц последнего заказа"] = last_o["date"].dt.month
new["календарь|доля заказов 1–10 числа"] = (o["date"].dt.day <= 10).groupby(o[cid]).mean()
new["календарь|доля заказов 21+ числа"] = (o["date"].dt.day >= 21).groupby(o[cid]).mean()
new["календарь|заказов в месяцы окна год назад"] = rel.between(10, 12).groupby(o[cid]).sum()
new["календарь|заказов в месяцы окна два года назад"] = rel.between(22, 24).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца перед окном год назад"] = rel.between(13, 15).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца после окна год назад"] = rel.between(7, 9).groupby(o[cid]).sum()
new["календарь|активных месяцев"] = mi.groupby(o[cid]).nunique()
new["календарь|активных месяцев за год"] = mi.where(rel <= 12).groupby(o[cid]).nunique()

NEW = pd.DataFrame(new).reindex(info.index).astype("float32")
NAMES = {f"n{i}": c for i, c in enumerate(NEW.columns)}       # короткие имена для моделей -> понятные имена
NEW.columns = list(NAMES)
data = data.join(NEW, on=cid)
BASE = [c for c in feats.columns if c != "top_brand"]
print(f"новых признаков: {NEW.shape[1]} | {(time.time() - T0) / 60:.1f} мин")

## Отбор новых признаков

In [ ]:
# Новый признак остаётся, если его важность выше 90-го процентиля важности на перемешанном таргете
trd = data[data["part"].eq("train")]
Xs, ys = trd[BASE + list(NAMES)], trd["target"].astype(int).to_numpy()
sel = dict(objective="binary", learning_rate=0.05, n_estimators=300, num_leaves=31, min_child_samples=100,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, n_jobs=-1, verbose=-1)


def gain(target, seed):
    m = lgb.LGBMClassifier(**sel, random_state=seed).fit(Xs, target)
    return pd.Series(m.booster_.feature_importance("gain"), index=Xs.columns)


rng = np.random.default_rng(SEED)
real = pd.concat([gain(ys, s) for s in range(3)], axis=1).mean(axis=1)
null = pd.concat([gain(rng.permutation(ys), s) for s in range(6)], axis=1)
ratio = real / (null.quantile(0.9, axis=1) + 1e-9)
KEEP_NEW = [c for c in NAMES if ratio[c] > 1.0]
FEATURES = BASE + KEEP_NEW
print(f"новых признаков выше шума: {len(KEEP_NEW)} из {len(NAMES)} | всего признаков: {len(FEATURES)} | {(time.time() - T0) / 60:.1f} мин")
print("Самые важные из новых:")
display((real[KEEP_NEW] / real.sum()).sort_values(ascending=False).head(20).rename(index=NAMES).to_frame("доля важности"))

## Банк признаков

In [ ]:
IDX = info.index
L = lines
OT = (L.groupby([cid, "order_id"]).agg(d=("d", "first"), s=("paid_price", "sum"), n=("paid_price", "size"),
                                       disc=("discount_percent", "mean"), loy=("loyalty_percent", "max"))
      .reset_index().sort_values([cid, "d", "order_id"]).reset_index(drop=True))
OT["k"] = OT.groupby(cid).cumcount()                 # 0 — самый свежий заказ
OT["gap"] = OT.groupby(cid)["d"].diff()              # пауза до соседнего более свежего заказа
R = OT.groupby(cid)["d"].min().reindex(IDX)
BANK = {}


def put(group, name, s):
    BANK.setdefault(group, {})[name] = pd.Series(s).reindex(IDX)


def ratio(a, b):
    return a / b.replace(0, np.nan)


# окна по времени и тренды между соседними окнами
W = [30, 60, 90, 120, 180, 270, 365, 540, 730]
cnt, summ = {}, {}
for w in W:
    m = OT["d"] < w
    cnt[w] = m.groupby(OT[cid]).sum().reindex(IDX).fillna(0)
    summ[w] = OT["s"].where(m, 0).groupby(OT[cid]).sum().reindex(IDX).fillna(0)
    put("окна", f"заказов <{w}", cnt[w])
    put("окна", f"сумма <{w}", summ[w])
    put("окна", f"позиций <{w}", OT["n"].where(m, 0).groupby(OT[cid]).sum())
    put("окна", f"скидка <{w}", OT["disc"].where(m).groupby(OT[cid]).mean())
for a, b in zip(W[:-1], W[1:]):
    put("окна", f"тренд заказов {a}/{b}", ratio(cnt[a], cnt[b] - cnt[a] + 1))
    put("окна", f"тренд суммы {a}/{b}", ratio(summ[a], summ[b] - summ[a] + 1))
    put("окна", f"доля заказов {a} из {b}", ratio(cnt[a], cnt[b]))

# последние k заказов
for k in [2, 3, 5, 10]:
    sub = OT[OT["k"] < k]
    g = sub.groupby(cid)
    c = g.size()
    span = g["d"].max() - g["d"].min()
    put("последние k", f"сумма ср {k}", g["s"].mean())
    put("последние k", f"позиций ср {k}", g["n"].mean())
    put("последние k", f"скидка ср {k}", g["disc"].mean())
    put("последние k", f"размах {k}", span.where(c == k))
    put("последние k", f"пауза ср {k}", (span / (c - 1).replace(0, np.nan)).where(c == k))
    put("последние k", f"давность {k}-го", g["d"].max().where(c == k))
    put("последние k", f"давность / пауза {k}", ratio(R, (span / (c - 1).replace(0, np.nan)).where(c == k).reindex(IDX)))

# статистики пауз по окнам
gp = OT[OT["gap"].notna()]
for H in [90, 180, 365, 730]:
    g = gp[gp["d"] < H].groupby(cid)["gap"]
    mean, med = g.mean().reindex(IDX), g.median().reindex(IDX)
    put("паузы", f"ср <{H}", mean)
    put("паузы", f"медиана <{H}", med)
    put("паузы", f"разброс <{H}", g.std())
    put("паузы", f"мин <{H}", g.min())
    put("паузы", f"макс <{H}", g.max())
    put("паузы", f"асимметрия <{H}", g.skew())
    for q in [7, 30, 90]:
        put("паузы", f"доля ≤{q} <{H}", (gp["gap"] <= q).where(gp["d"] < H).groupby(gp[cid]).mean())
    put("паузы", f"давность - ср <{H}", R - mean)
    put("паузы", f"давность / медиана <{H}", ratio(R, med))
    put("паузы", f"макс - давность <{H}", g.max().reindex(IDX) - R)

# повторы товаров, брендов и подкатегорий
L2 = L[[cid, "d", "product_id", "brand", "category2", "category"]].copy()
L2["last"] = L2["d"].eq(L2[cid].map(R))
for key in ["product_id", "brand", "category2", "category"]:
    u = L2.dropna(subset=[key]).drop_duplicates([cid, "d", key])
    days_per = u.groupby([cid, key]).size()
    put("повторы", f"{key}: доля повторных", (days_per >= 2).groupby(level=0).mean())
    put("повторы", f"{key}: макс дней", days_per.groupby(level=0).max())
    before = u[~u["last"]][[cid, key]].drop_duplicates().assign(seen=1)
    lastk = u[u["last"]][[cid, key]].drop_duplicates().merge(before, on=[cid, key], how="left")
    put("повторы", f"{key}: последний заказ уже покупал", lastk["seen"].fillna(0).groupby(lastk[cid]).mean())

# денежные тренды
for a, b in [(90, 180), (180, 365), (365, 730)]:
    put("деньги", f"сумма {a} / {a}-{b}", ratio(summ[a], summ[b] - summ[a]))
put("деньги", "средний чек 365 / 730", ratio(ratio(summ[365], cnt[365]), ratio(summ[730], cnt[730])))
put("деньги", "чек последнего / средний", ratio(OT[OT["k"] == 0].set_index(cid)["s"].reindex(IDX), OT.groupby(cid)["s"].mean().reindex(IDX)))
put("деньги", "лояльность сейчас - год назад", OT[OT["k"] == 0].set_index(cid)["loy"].reindex(IDX)
    - OT[OT["d"] >= 300].groupby(cid)["loy"].first().reindex(IDX))
put("деньги", "скидка 90 - скидка 365", OT["disc"].where(OT["d"] < 90).groupby(OT[cid]).mean().reindex(IDX)
    - OT["disc"].where(OT["d"] < 365).groupby(OT[cid]).mean().reindex(IDX))

# давность и частота по каждой категории
for c in L["category"].dropna().unique():
    s = L[L["category"] == c]
    dd = s.drop_duplicates([cid, "d"])
    put("категории", f"{c}: давность", dd.groupby(cid)["d"].min())
    put("категории", f"{c}: дней <365", (dd["d"] < 365).groupby(dd[cid]).sum())
    put("категории", f"{c}: доля позиций <180", ratio((s["d"] < 180).groupby(s[cid]).sum().reindex(IDX),
                                                       (L["d"] < 180).groupby(L[cid]).sum().reindex(IDX)))

# бренды
b365 = L[L["d"] < 365]
put("бренды", "брендов <365", b365.groupby(cid)["brand"].nunique())
put("бренды", "брендов <90", L[L["d"] < 90].groupby(cid)["brand"].nunique())
put("бренды", "брендов на заказ", ratio(L.groupby(cid)["brand"].nunique().reindex(IDX), OT.groupby(cid).size().reindex(IDX)))
topb = L.groupby([cid, "brand"]).size()
put("бренды", "доля главного бренда", topb.groupby(level=0).max() / topb.groupby(level=0).sum())
put("бренды", "подкатегорий <365", b365.groupby(cid)["category2"].nunique())

GROUPS, BANK_NAMES = {}, {}
for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
print("групп:", len(GROUPS), "| признаков в банке:", len(BANK_NAMES), {g: len(c) for g, c in GROUPS.items()}, f"| {(time.time() - T0) / 60:.1f} мин")

## Паузы внутри категорий

In [ ]:
BANK = {}
for col in ["category", "category2"]:
    dd = L[[cid, col, "d"]].dropna().drop_duplicates().sort_values([cid, col, "d"])
    dd["gap"] = dd.groupby([cid, col])["d"].diff()
    st = dd.groupby([cid, col]).agg(last=("d", "min"), n=("d", "size"), med=("gap", "median"), mx=("gap", "max")).reset_index()
    st = st[st["n"] >= 2]
    st["r_med"] = st["last"] / st["med"].replace(0, np.nan)
    st["med_r"] = st["med"] - st["last"]
    st["mx_r"] = st["mx"] - st["last"]
    g = st.groupby(cid)
    put("паузы в категориях", f"{col}: мин давность / медиана", g["r_med"].min())
    put("паузы в категориях", f"{col}: ср давность / медиана", g["r_med"].mean())
    put("паузы в категориях", f"{col}: медиана - давность, макс", g["med_r"].max())
    put("паузы в категориях", f"{col}: медиана - давность, мин |.|", st.assign(a=st["med_r"].abs()).groupby(cid)["a"].min())
    put("паузы в категориях", f"{col}: макс пауза - давность, макс", g["mx_r"].max())
    put("паузы в категориях", f"{col}: серий 3+ дней", (st["n"] >= 3).groupby(st[cid]).sum())
    top = st.sort_values([cid, "n", "last"], ascending=[True, False, True]).drop_duplicates(cid).set_index(cid)
    put("паузы в категориях", f"{col} главная: медиана", top["med"])
    put("паузы в категориях", f"{col} главная: давность", top["last"])
    put("паузы в категориях", f"{col} главная: давность / медиана", top["r_med"])

for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
FEATURES_V2 = FEATURES
FEATURES = FEATURES_V2 + GROUPS["паузы"] + GROUPS["категории"] + GROUPS["паузы в категориях"]
print("признаков v2:", len(FEATURES_V2), "| с банком:", len(FEATURES), f"| {(time.time() - T0) / 60:.1f} мин")

## Паузы по каждой категории и окно по категориям

In [ ]:
FEATURES_V4 = FEATURES
BANK = {}
HI_ = GAP_DAYS + info["win_len"].astype(float)       # целевое окно в днях после отсечения: (GAP_DAYS, HI_]


def cycles(col, src=None):
    # по каждой паре (клиент, значение col): давность последней покупки, первой, число дней, медиана и максимум паузы
    src = L if src is None else src
    dd = src[[cid, col, "d"]].dropna().drop_duplicates().sort_values([cid, col, "d"])
    dd["gap"] = dd.groupby([cid, col])["d"].diff()
    st = dd.groupby([cid, col]).agg(last=("d", "min"), first=("d", "max"), n=("d", "size"), med=("gap", "median"),
                                     mx=("gap", "max")).reset_index()
    st = st[st["n"] >= 2].copy()
    st["r_med"] = st["last"] / st["med"].replace(0, np.nan)
    st["t_med"] = st["med"] - st["last"]                                  # ожидаемая следующая покупка по медиане
    st["t_mean"] = (st["first"] - st["last"]) / (st["n"] - 1) - st["last"]  # то же по средней паузе
    return st


# паузы отдельно по каждой категории и полу товара
for col in ["category", "gender_product"]:
    st = cycles(col)
    for c in L[col].dropna().unique():
        s = st[st[col] == c].set_index(cid)
        put("паузы по каждой категории", f"{col} {c}: медиана", s["med"])
        put("паузы по каждой категории", f"{col} {c}: давность / медиана", s["r_med"])
        put("паузы по каждой категории", f"{col} {c}: медиана - давность", s["t_med"])
        put("паузы по каждой категории", f"{col} {c}: средняя - давность", s["t_mean"])
        put("паузы по каждой категории", f"{col} {c}: макс пауза", s["mx"])

# попадает ли ожидаемая покупка по категориям в целевое окно
for col in ["category", "category2", "brand"]:
    st = cycles(col)
    hi = st[cid].map(HI_)
    mid = (GAP_DAYS + hi) / 2
    for t in ["t_med", "t_mean"]:
        inn = ((st[t] > GAP_DAYS) & (st[t] <= hi)).astype(float)
        dist = (GAP_DAYS - st[t]).clip(lower=0) + (st[t] - hi).clip(lower=0)
        put("окно по категориям", f"{col} {t}: в окне", inn.groupby(st[cid]).sum())
        put("окно по категориям", f"{col} {t}: доля в окне", inn.groupby(st[cid]).mean())
        put("окно по категориям", f"{col} {t}: доля в окне, вес по дням", (inn * st["n"]).groupby(st[cid]).sum() / st["n"].groupby(st[cid]).sum())
        put("окно по категориям", f"{col} {t}: мин расстояние", dist.groupby(st[cid]).min())
        put("окно по категориям", f"{col} {t}: мин |t - середина окна|", (st[t] - mid).abs().groupby(st[cid]).min())
    put("окно по категориям", f"{col}: t_mean в (35, 85]", st["t_mean"].between(35, 85, inclusive="right").groupby(st[cid]).sum())

for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
FEATURES = FEATURES_V4 + GROUPS["паузы по каждой категории"] + GROUPS["окно по категориям"]
print("признаков v4:", len(FEATURES_V4), "| v5:", len(FEATURES), f"| {(time.time() - T0) / 60:.1f} мин")

## Перепроверяемые и новые группы

In [ ]:
BASE5 = FEATURES
BANK = {}
rel = L[cid].map(month_idx(info["scoring_date"])) - month_idx(L["order_date"])   # месяцев от покупки до начала окна
LD = L.assign(rel=rel).drop_duplicates([cid, "d", "category", "gender_product"])

# сезонность: покупки категории в те же месяцы год и два назад
for col in ["category", "gender_product"]:
    for c in L[col].dropna().unique():
        s = LD[LD[col] == c].drop_duplicates([cid, "d"])
        put("сезонность категорий", f"{col} {c}: дней в месяцы окна год назад", s["rel"].between(10, 12).groupby(s[cid]).sum())
        put("сезонность категорий", f"{col} {c}: дней в месяцы окна два года назад", s["rel"].between(22, 24).groupby(s[cid]).sum())
        put("сезонность категорий", f"{col} {c}: доля дней в месяцы окна", s["rel"].mod(12).isin([10, 11, 0]).groupby(s[cid]).mean())
    u = LD.dropna(subset=[col]).drop_duplicates([cid, col, "rel"])
    put("сезонность категорий", f"{col}: категорий в месяцы окна год назад", u[u["rel"].between(10, 12)].groupby(cid)[col].nunique())
    put("сезонность категорий", f"{col}: категорий в месяцы окна два года назад", u[u["rel"].between(22, 24)].groupby(cid)[col].nunique())
c2 = L.assign(rel=rel).dropna(subset=["category2"]).drop_duplicates([cid, "category2", "rel"])
put("сезонность категорий", "подкатегорий в месяцы окна год назад", c2[c2["rel"].between(10, 12)].groupby(cid)["category2"].nunique())
put("сезонность категорий", "подкатегорий год назад, купленных и позже", c2[c2["rel"].between(10, 12)]
    .merge(c2[c2["rel"] < 10][[cid, "category2"]].drop_duplicates(), on=[cid, "category2"]).groupby(cid)["category2"].nunique())

# циклы категорий только за последний год
LY = L[L["d"] < 365]
for col in ["category", "category2", "gender_product"]:
    st = cycles(col, LY)
    g = st.groupby(cid)
    put("циклы за год", f"{col}: серий", g.size())
    put("циклы за год", f"{col}: мин давность / медиана", g["r_med"].min())
    put("циклы за год", f"{col}: ср давность / медиана", g["r_med"].mean())
    put("циклы за год", f"{col}: медиана - давность, макс", g["t_med"].max())
    put("циклы за год", f"{col}: медиана - давность, мин |.|", st.assign(a=st["t_med"].abs()).groupby(cid)["a"].min())
    put("циклы за год", f"{col}: в окне", ((st["t_med"] > GAP_DAYS) & (st["t_med"] <= st[cid].map(HI_))).groupby(st[cid]).sum())
    top = st.sort_values([cid, "n", "last"], ascending=[True, False, True]).drop_duplicates(cid).set_index(cid)
    put("циклы за год", f"{col} главная: медиана", top["med"])
    put("циклы за год", f"{col} главная: давность / медиана", top["r_med"])

# категории последнего дня против остальных
last_day = L[L["d"].eq(L[cid].map(R))]
for col in ["category", "category2"]:
    st = cycles(col)
    inl = last_day[[cid, col]].dropna().drop_duplicates().assign(in_last=1)
    st = st.merge(inl, on=[cid, col], how="left")
    st["in_last"] = st["in_last"].fillna(0).astype(bool)
    o_ = st[~st["in_last"]]
    put("последний день", f"{col}: других категорий с циклом", o_.groupby(cid).size())
    put("последний день", f"{col}: другие, мин |медиана - давность|", o_["t_med"].abs().groupby(o_[cid]).min())
    put("последний день", f"{col}: другие, макс давность / медиана", o_.groupby(cid)["r_med"].max())
    put("последний день", f"{col}: другие в окне", ((o_["t_med"] > GAP_DAYS) & (o_["t_med"] <= o_[cid].map(HI_))).groupby(o_[cid]).sum())
    i_ = st[st["in_last"]]
    put("последний день", f"{col}: купленные в последний день, ср медиана", i_.groupby(cid)["med"].mean())
    put("последний день", f"{col}: купленные в последний день, мин медиана", i_.groupby(cid)["med"].min())
    put("последний день", f"{col}: категорий в последний день", last_day.groupby(cid)[col].nunique())

# формула t по категориям и полу товара: как в правиле для 2–3 заказов, но внутри категории
for col in ["category", "gender_product"]:
    dd = L[[cid, col, "d"]].dropna().drop_duplicates().sort_values([cid, col, "d"])
    dd["rk"] = dd.groupby([cid, col]).cumcount()
    for tag, part in [("все", dd), ("3 последних", dd[dd["rk"] < 3])]:
        st = part.groupby([cid, col])["d"].agg(n="size", last="min", first="max").reset_index()
        st["t"] = (st["first"] - st["last"]) / (st["n"] - 1).replace(0, np.nan) - st["last"]
        st["in"] = st["t"].between(35, 85, inclusive="right").astype(float).where(st["t"].notna())
        for c in dd[col].unique():
            s = st[st[col] == c].set_index(cid)
            put("t по категориям", f"{col} {c} {tag}: t", s["t"])
            if tag == "все":
                put("t по категориям", f"{col} {c}: дней", s["n"])
        put("t по категориям", f"{col} {tag}: категорий с t в окне", st.groupby(cid)["in"].sum())
        put("t по категориям", f"{col} {tag}: доля с t в окне", st.groupby(cid)["in"].mean())
        put("t по категориям", f"{col} {tag}: t ближе всего к 60", st.assign(a=(st["t"] - 60).abs()).sort_values([cid, "a"])
            .drop_duplicates(cid).set_index(cid)["t"])


# история прогнозов: что было в прошлых 3-месячных окнах и предсказывала ли их формула t по заказам до того момента
OD = OT[[cid, "d", "s"]].copy()
OD["date"] = OD[cid].map(info["cutoff"]) - pd.to_timedelta(OD["d"], unit="D")
SD = info["scoring_date"]


def shifted(months):
    # дата оценки, сдвинутая на months месяцев назад, — для каждой строки OD (считаем по клиентам, потом раскладываем)
    return OD[cid].map(SD - pd.DateOffset(months=months))


hits, ins = [], []
for k in [4, 6, 8, 12]:
    sd_k = shifted(k)
    cut_k = sd_k - pd.Timedelta(days=GAP_DAYS)
    win = (OD["date"] >= sd_k) & (OD["date"] < shifted(k - TARGET_MONTHS))
    hist = OD["date"] <= cut_k
    dk = (cut_k - OD["date"]).dt.days.where(hist)
    g = dk.groupby(OD[cid])
    n_k, last_k, first_k = g.count(), g.min(), g.max()
    n_k = n_k.where(n_k > 0)
    t_k = ((first_k - last_k) / (n_k - 1).replace(0, np.nan) - last_k).reindex(IDX)
    hit = win.groupby(OD[cid]).any().astype(float).reindex(IDX).where(n_k.reindex(IDX).notna())
    inn = t_k.between(35, 85, inclusive="right").astype(float).where(t_k.notna())
    put("история прогнозов", f"{k} мес назад: покупал в окне", hit)
    put("история прогнозов", f"{k} мес назад: t", t_k)
    put("история прогнозов", f"{k} мес назад: давность", last_k)
    put("история прогнозов", f"{k} мес назад: t в (35, 85] и покупал", inn * hit)
    put("история прогнозов", f"{k} мес назад: t в (35, 85] и не покупал", inn * (1 - hit))
    hits.append(hit)
    ins.append(inn)
H, I = pd.concat(hits, axis=1), pd.concat(ins, axis=1)
put("история прогнозов", "доля окон с покупкой", H.mean(axis=1))
put("история прогнозов", "окон с покупкой", H.sum(axis=1, min_count=1))
put("история прогнозов", "доля верных прогнозов t", (H == I).astype(float).where(H.notna() & I.notna()).mean(axis=1))

# весна год назад: окно тех же месяцев и три месяца до и после него, как вёл себя клиент тогда
s12, s15, s9, s6 = shifted(12), shifted(15), shifted(9), shifted(6)
ya_win = (OD["date"] >= s12) & (OD["date"] < s9)
for name, m in [("окно", ya_win), ("до окна", (OD["date"] >= s15) & (OD["date"] < s12)), ("после окна", (OD["date"] >= s9) & (OD["date"] < s6))]:
    put("весна год назад", f"заказов: {name}", m.groupby(OD[cid]).sum())
    put("весна год назад", f"сумма: {name}", OD["s"].where(m, 0).groupby(OD[cid]).sum())
w = OD[ya_win].sort_values([cid, "d"])
put("весна год назад", "медиана паузы в окне", w.groupby(cid)["d"].diff().groupby(w[cid]).median())
put("весна год назад", "первый заказ в окне, дней от начала окна", (w["date"] - s12[w.index]).dt.days.groupby(w[cid]).min())
put("весна год назад", "доля заказов в месяцы окна за всю историю",
    (OD["date"].dt.month - OD[cid].map(SD.dt.month)).mod(12).lt(TARGET_MONTHS).groupby(OD[cid]).mean())

GROUPS_ADD = {"повторы": GROUPS["повторы"]}
for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS_ADD[gname] = short
GROUPS_DROP = {"новые признаки v2": KEEP_NEW, **{g: GROUPS[g] for g in ["паузы", "категории", "паузы в категориях",
                                                                         "паузы по каждой категории", "окно по категориям"]}}
print("база v5:", len(BASE5), "| убираем по одной:", {g: len(c) for g, c in GROUPS_DROP.items()},
      "| добавляем по одной:", {g: len(c) for g, c in GROUPS_ADD.items()}, f"| {(time.time() - T0) / 60:.1f} мин")

## Чистка и добавление по марту

In [ ]:
tr = data[data["part"].eq("train")].reset_index(drop=True)
te = data[data["part"].eq("test")].set_index(cid).loc[sample[cid]].reset_index()
y, sl = tr["target"].astype(int).to_numpy(), tr["slice"].to_numpy()
P = dict(objective="binary", learning_rate=0.05, n_estimators=3000, subsample_freq=1, num_leaves=47, min_child_samples=85,
         colsample_bytree=0.6, subsample=0.89, reg_lambda=0.66, min_split_gain=0.43, n_jobs=-1, verbose=-1)
MAR = np.where(sl == "2022-03-01")[0]
JF = np.where(sl != "2022-03-01")[0]
yb = y[MAR]
BOOT = [np.random.default_rng(i).integers(0, len(MAR), len(MAR)) for i in range(300)]
SEEDS = [0, 1, 2]
N_MAR = tr["n_orders"].iloc[MAR].to_numpy()


def march(cols):
    pred = np.zeros(len(MAR))
    for s in SEEDS:
        perm = np.random.default_rng(SEED + s).permutation(JF)
        aa, es = perm[: int(len(JF) * 0.9)], perm[int(len(JF) * 0.9):]
        m = lgb.LGBMClassifier(**P, random_state=SEED + s).fit(tr[cols].iloc[aa], y[aa], eval_set=[(tr[cols].iloc[es], y[es])],
                                                             callbacks=[lgb.early_stopping(100, verbose=False)])
        pred += m.predict_proba(tr[cols].iloc[MAR])[:, 1] / len(SEEDS)
    return pred


def paired(p, q):
    d = np.array([fast_ap(yb[i], p[i]) - fast_ap(yb[i], q[i]) for i in BOOT])
    return fast_ap(yb, p) - fast_ap(yb, q), d.std(ddof=1)


def row(name, cols, pred, ref):
    d, se = paired(pred, ref)
    return {"набор": name, "признаков": len(cols), "AP@20% март": fast_ap(yb, pred), "прирост": d, "шум": se, "прирост / шум": d / se,
            "AUC 1": roc_auc_score(yb[N_MAR == 1], pred[N_MAR == 1]), "AUC 2-3": roc_auc_score(yb[(N_MAR >= 2) & (N_MAR <= 3)], pred[(N_MAR >= 2) & (N_MAR <= 3)]),
            "AUC 4+": roc_auc_score(yb[N_MAR >= 4], pred[N_MAR >= 4])}


base_pred = march(BASE5)
print(f"v5 на марте: {fast_ap(yb, base_pred):.4f} | {(time.time() - T0) / 60:.1f} мин", flush=True)
rows = []
for gname, cols in GROUPS_DROP.items():
    c = [x for x in BASE5 if x not in set(cols)]
    rows.append({"действие": "убрать", "группа": gname, **row(gname, c, march(c), base_pred)})
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rows[-1].items()}, flush=True)
for gname, cols in GROUPS_ADD.items():
    c = BASE5 + cols
    rows.append({"действие": "добавить", "группа": gname, **row(gname, c, march(c), base_pred)})
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rows[-1].items()}, flush=True)
sel = pd.DataFrame(rows).drop(columns="набор").set_index(["действие", "группа"])
# убираем группу, если без неё март лучше больше чем на один шум; добавляем, если с ней март лучше больше чем на два шума
sel["решение"] = np.where(sel.index.get_level_values(0) == "убрать",
                          np.where(sel["прирост"] > sel["шум"], "убрать", "оставить"),
                          np.where((sel["прирост"] > 2 * sel["шум"]) & (sel["прирост"] > 0.001), "добавить", "не брать"))
display(sel)

## Поиск правил

In [ ]:
# Неглубокие деревья по понятным величинам отдельно для каждой группы по числу заказов. Ищем листья, где почти все
# покупают или почти никто, на январе и феврале, и смотрим, держатся ли они на марте — так нашлось бы правило вроде 2–3 заказов.
from sklearn.tree import DecisionTreeClassifier

INV = {v: k for k, v in NAMES.items()}

RULE_COLS = ["recency", "tenure", "n_orders", "n_days", "gap_mean", "gap_median", "gap_last", "gap_min", "gap_max", "gap_std",
             "n_items", "sum_paid", "order_sum_mean", "items_per_order", "max_day_orders", "gap_prev", "d_day_2", "d_day_3",
             INV["заказ|вся|mean|t"], INV["заказ|год|max|t"], INV["заказ|вся|median|t"], INV["заказ|год|q75|t"]]
RULE_NAMES = {c: NAMES.get(c, c) for c in RULE_COLS}
nn = tr["n_orders"].to_numpy()
found = []
for gname, m in [("1", nn == 1), ("2", nn == 2), ("3", nn == 3), ("4-5", (nn >= 4) & (nn <= 5)), ("6-9", (nn >= 6) & (nn <= 9)), ("10+", nn >= 10)]:
    a, b = np.intersect1d(np.where(m)[0], JF), np.intersect1d(np.where(m)[0], MAR)
    Xa = tr[RULE_COLS].iloc[a].fillna(-999)
    for depth in [2, 3, 4]:
        t = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=100, random_state=SEED).fit(Xa, y[a])
        la, lb = t.apply(Xa), t.apply(tr[RULE_COLS].iloc[b].fillna(-999))
        tree = t.tree_

        def path(leaf):
            # условия на пути от корня к листу
            parent = {c: (i, "<=") for i, c in enumerate(tree.children_left) if c != -1}
            parent.update({c: (i, ">") for i, c in enumerate(tree.children_right) if c != -1})
            conds, node = [], leaf
            while node in parent:
                i, op = parent[node]
                conds.append(f"{RULE_NAMES[RULE_COLS[tree.feature[i]]]} {op} {tree.threshold[i]:.1f}")
                node = i
            return " и ".join(reversed(conds))

        for leaf in np.unique(la):
            ra, rb = y[a][la == leaf], y[b][lb == leaf]
            if len(ra) >= 100 and (ra.mean() >= 0.9 or ra.mean() <= 0.05):
                found.append({"заказов": gname, "глубина": depth, "правило": path(leaf), "клиентов янв+фев": len(ra),
                              "доля покупателей янв+фев": ra.mean(), "клиентов март": len(rb),
                              "доля покупателей март": rb.mean() if len(rb) else np.nan, "доля группы": len(ra) / len(a)})
rules = pd.DataFrame(found)
if len(rules):
    rules = rules.drop_duplicates("правило").sort_values(["заказов", "доля покупателей янв+фев"], ascending=[True, False])
    pd.set_option("display.max_colwidth", 200)
    display(rules)
else:
    print("резких листьев не нашлось")

## Итоговый набор

In [ ]:
drop = [g for (act, g), r in sel.iterrows() if r["решение"] == "убрать"]
add = [g for (act, g), r in sel.iterrows() if r["решение"] == "добавить"]
drop_cols = set(itertools.chain.from_iterable(GROUPS_DROP[g] for g in drop))
FINAL = [c for c in BASE5 if c not in drop_cols] + list(itertools.chain.from_iterable(GROUPS_ADD[g] for g in add))
print("убираем:", drop or "ничего", "| добавляем:", add or "ничего", "| признаков:", len(FINAL))
if drop or add:
    fin_pred = march(FINAL)
    r = row("итог", FINAL, fin_pred, base_pred)
    print(f"март: v5 {fast_ap(yb, base_pred):.4f} → итог {r['AP@20% март']:.4f}, прирост {r['прирост']:+.4f} ± {r['шум']:.4f}")
    strata = tr["slice"] + "_" + tr["target"].astype(int).astype(str)
    SPLITS = [list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + s).split(tr, strata)) for s in range(2)]

    def run(cols):
        out = []
        for split in SPLITS:
            for a, b in split:
                m = lgb.LGBMClassifier(**P, random_state=SEED).fit(tr[cols].iloc[a], y[a], eval_set=[(tr[cols].iloc[b], y[b])],
                                                                 callbacks=[lgb.early_stopping(100, verbose=False)])
                p = m.predict_proba(tr[cols].iloc[b])[:, 1]
                out.append(np.mean([fast_ap(y[b][sl[b] == s], p[sl[b] == s]) for s in np.unique(sl[b])]))
        return np.array(out)

    cv_b, cv_f = run(BASE5), run(FINAL)
    dd = cv_f - cv_b
    print(f"CV: v5 {cv_b.mean():.4f} → итог {cv_f.mean():.4f}, прирост {dd.mean():+.4f} ± {dd.std(ddof=1) / np.sqrt(len(dd)):.4f}")
    rows = []
    for s in np.unique(sl):
        a, b = np.where(sl != s)[0], np.where(sl == s)[0]
        res = {"проверочный срез": s}
        for name, cols in [("v5", BASE5), ("итог", FINAL)]:
            perm = np.random.default_rng(SEED).permutation(a)
            aa, es = perm[: int(len(a) * 0.9)], perm[int(len(a) * 0.9):]
            m = lgb.LGBMClassifier(**P, random_state=SEED).fit(tr[cols].iloc[aa], y[aa], eval_set=[(tr[cols].iloc[es], y[es])],
                                                             callbacks=[lgb.early_stopping(100, verbose=False)])
            res[name] = fast_ap(y[b], m.predict_proba(tr[cols].iloc[b])[:, 1])
        res["прирост"] = res["итог"] - res["v5"]
        rows.append(res)
    print("Перенос во времени (одно зерно):")
    display(pd.DataFrame(rows).set_index("проверочный срез"))
else:
    print("набор не меняется, итог = v5")

test = np.zeros(len(te))
strata = tr["slice"] + "_" + tr["target"].astype(int).astype(str)
for a, b in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(tr, strata):
    m = lgb.LGBMClassifier(**P, random_state=SEED).fit(tr[FINAL].iloc[a], y[a], eval_set=[(tr[FINAL].iloc[b], y[b])],
                                                     callbacks=[lgb.early_stopping(100, verbose=False)])
    test += m.predict_proba(te[FINAL])[:, 1] / N_FOLDS
sub = pd.DataFrame({cid: te[cid].to_numpy(), "score": pd.Series(test).rank(pct=True).to_numpy()})
assert sub[cid].tolist() == sample[cid].tolist()
sub.to_csv(OUT_DIR / "submission_march_lgb.csv", index=False)
json.dump({"drop": drop, "add": add, "selection": sel.reset_index().to_dict("records"),
           "rules": rules.to_dict("records") if len(rules) else []},
          open(OUT_DIR / "features_march.json", "w"), ensure_ascii=False, indent=1, default=str)
print(f"Сохранено: submission_march_lgb.csv, features_march.json. Всего времени: {(time.time() - T0) / 60:.1f} мин")